In [2]:
# Load and clean TTC bus delay data directly from the Toronto Open Data CKAN API.

import os
import re
import sys
import requests
from io import BytesIO
from urllib.parse import urlsplit

import pandas as pd

#Normalise column names
COLUMN_MAP = {
    "report date": "Date",
    "date": "Date",
    "line": "Route",          # Jul-Aug 2021
    "route": "Route",
    "time": "Time",
    "day": "Day",
    "location": "Station",
    "incident": "Incident",
    "incident id": "Incident ID",   # Apr 2019 only
    "min delay": "Min Delay",
    "delay": "Min Delay",
    "min gap": "Min Gap",
    "gap": "Min Gap",
    "direction": "Direction",
    "bound": "Direction",     # Jul-Aug 2021
    "vehicle": "Vehicle",
}
FINAL_COLS = ["Date", "Time", "Day", "Route", "Station", "Incident",
              "Incident Raw", "Min Delay", "Min Gap", "Direction", "Vehicle"]

INCIDENT_FIXES = {
    "Securitty": "Security",
    "Utilizing Off Route": "Utilized Off Route",
    "Collision - TTC Involved": "Collision - TTC",
    "Road Block - Non-TTC Collision": "Road Blocked - Non-TTC Collision",
    "Road Blocked - NON-TTC Collision": "Road Blocked - Non-TTC Collision",
    "Roadblock by Collision - Non-TTC": "Road Blocked - Non-TTC Collision",
    "Operations - Operator": "Operations",
    "Held By": "Held By",
}

INCIDENT_PREFIXES = ["Late Leaving Garage", "Late Entering Service", "Cleaning"]
INCIDENT_JUNK = {"e", ""}  

# Fixed direction codes
DIRECTION_MAP = {
    "N": "N", "NB": "N", "N/B": "N", "NORTH": "N",
    "S": "S", "SB": "S", "S/B": "S", "SOUTH": "S",
    "E": "E", "EB": "E", "E/B": "E", "EAST": "E",
    "W": "W", "WB": "W", "W/B": "W", "WEST": "W",
    "B": "B", "BW": "B", "B/W": "B", "BOTH": "B", "BOTH WAYS": "B",
}


def _clean_incident(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.mask(s.str.lower().isin(INCIDENT_JUNK))
    s = s.replace(INCIDENT_FIXES)
    for p in INCIDENT_PREFIXES:
        s = s.mask(s.str.startswith(p, na=False), p)
    return s


def _clean_direction(s: pd.Series) -> pd.Series:
    key = s.astype("string").str.strip().str.upper()
    return key.map(DIRECTION_MAP).astype("string")   


def _parse_time(s: pd.Series) -> pd.Series:
    def one(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return pd.NA
        if hasattr(v, "hour"):
            return f"{v.hour:02d}:{v.minute:02d}"
        m = re.match(r"^\s*(\d{1,2}):(\d{2})", str(v))
        return f"{int(m.group(1)):02d}:{m.group(2)}" if m else pd.NA
    return s.map(one).astype("string")


def _normalize_resource(raw_df: pd.DataFrame, source_name: str, sheet_name: str) -> pd.DataFrame | None:
    keep = {}
    for column in raw_df.columns:
        key = str(column).strip().lower()
        if key in COLUMN_MAP:
            keep[column] = COLUMN_MAP[key]

    if "Date" not in keep.values():
        return None

    df = raw_df.loc[:, list(keep)].rename(columns=keep)

    for column in ["Time", "Day", "Route", "Station", "Incident",
                   "Min Delay", "Min Gap", "Direction", "Vehicle"]:
        if column not in df.columns:
            df[column] = pd.NA

    df = df.dropna(subset=["Date"])
    df["Source File"] = source_name
    df["Sheet"] = sheet_name
    return df


# Toronto Open Data is hosted on CKAN: https://docs.ckan.org/en/latest/api/
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

package_response = requests.get(
    f"{base_url}/api/3/action/package_show",
    params={"id": "ttc-bus-delay-data"},
    timeout=30,
)
package_response.raise_for_status()
package_payload = package_response.json()
if not package_payload.get("success"):
    raise RuntimeError("CKAN could not retrieve the TTC bus delay dataset.")

api_frames = []
for resource in package_payload["result"]["resources"]:
    if resource.get("datastore_active"):
        continue

    resource_listing_name = resource.get("name", "")
    if any(int(year) >= 2025 for year in re.findall(r"\b20\d{2}\b", resource_listing_name)):
        continue

    metadata_response = requests.get(
        f"{base_url}/api/3/action/resource_show",
        params={"id": resource["id"]},
        timeout=30,
    )
    metadata_response.raise_for_status()
    metadata_payload = metadata_response.json()
    if not metadata_payload.get("success"):
        raise RuntimeError(f"CKAN could not retrieve resource {resource['id']!r}.")

    metadata = metadata_payload["result"]
    file_url = metadata["url"]
    resource_name = metadata["name"]
    resource_years = re.findall(
        r"\b20\d{2}\b", f"{resource_name} {urlsplit(file_url).path}"
    )
    if any(int(year) >= 2025 for year in resource_years):
        continue
    extension = os.path.splitext(urlsplit(file_url).path)[1].lower()
    if extension not in {".csv", ".xlsx"}:
        continue

    file_response = requests.get(file_url, timeout=60)
    file_response.raise_for_status()
    file_content = BytesIO(file_response.content)
    if extension == ".csv":
        sheets = {"CSV": pd.read_csv(file_content)}
    else:
        sheets = pd.read_excel(file_content, sheet_name=None)

    for sheet_name, raw_df in sheets.items():
        normalized = _normalize_resource(raw_df, resource_name, str(sheet_name).strip())
        if normalized is not None:
            api_frames.append(normalized)


def load_all() -> pd.DataFrame:
    if not api_frames:
        raise RuntimeError("The CKAN API returned no supported bus-delay files.")

    df = pd.concat(api_frames, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df[df["Date"] < pd.Timestamp("2025-01-01")]
    df["Time"] = _parse_time(df["Time"])
    df["Datetime"] = pd.to_datetime(
        df["Date"].dt.strftime("%Y-%m-%d") + " " + df["Time"].fillna("00:00"),
        errors="coerce",
    )

    df["Route"] = pd.to_numeric(df["Route"], errors="coerce").astype("Int64")
    df["Min Delay"] = pd.to_numeric(df["Min Delay"], errors="coerce")
    df["Min Gap"] = pd.to_numeric(df["Min Gap"], errors="coerce")
    df["Vehicle"] = pd.to_numeric(df["Vehicle"], errors="coerce").astype("Int64")
    df["Vehicle"] = df["Vehicle"].mask(df["Vehicle"] == 0)      # 0 = missing

    df["Incident Raw"] = df["Incident"].astype("string").str.strip()
    df["Incident"] = _clean_incident(df["Incident"])
    df["Direction"] = _clean_direction(df["Direction"])
    df["Day"] = df["Day"].astype("string").str.strip().str.title()
    df["Station"] = df["Station"].astype("string").str.strip()

    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month

    cols = ["Datetime", "Year", "Month"] + FINAL_COLS + ["Source File", "Sheet"]
    return df[cols].sort_values("Datetime").reset_index(drop=True)


if __name__ == "__main__":
    out = sys.argv[1] if len(sys.argv) > 1 and not sys.argv[1].startswith("-") else None
    data = load_all()
    print(data.shape)
    print(data.dtypes)
    print(data.groupby("Year").size())
    data.to_csv("pre_2025_ttc_bus_delays_cleaned.csv", index=False)
    print("wrote pre_2025_ttc_bus_delays_cleaned.csv")


(707398, 16)
Datetime        datetime64[us]
Year                     int32
Month                    int32
Date            datetime64[us]
Time                    string
Day                     string
Route                    Int64
Station                 string
Incident                string
Incident Raw            string
Min Delay              float64
Min Gap                float64
Direction               string
Vehicle                  Int64
Source File                str
Sheet                      str
dtype: object
Year
2014    94217
2015    76510
2016    77088
2017    70303
2018    73927
2019    62376
2020    36151
2021    42269
2022    58707
2023    56207
2024    59643
dtype: int64
wrote pre_2025_ttc_bus_delays_cleaned.csv
